# Matched full-fine-tuning control for the V9 LRAA study

This is a **single controlled experiment**, not a model-improvement run. It
changes only the backbone-training regime relative to frozen V9:

- same passage-grouped split and seeds `13, 42, 71`;
- same DeBERTa-v3-large base weights and input serialization;
- same rank-256, eight-head LRAA and 1,025-parameter scorer;
- same learning rate `2e-4`, three epochs, batch size 2, gradient
  accumulation 8, weight decay 0.01, and 6% warm-up;
- **the DeBERTa backbone is trainable**.

Each selected checkpoint is evaluated twice on the untouched 3,270-example
BoolQ held-out split: LRAA enabled and the same checkpoint with only the LRAA
residual branch disabled. The output compares this matched full-fine-tuning
ablation loss with the already archived frozen-V9 loss seed by seed.

Full 434M-parameter encoder checkpoints are intentionally not placed in the
downloadable ZIP because three copies would be several gigabytes. The ZIP
contains complete predictions, training history, small adapter/scorer states,
environment/configuration records, and integrity hashes; the notebook itself
is the executable retraining specification. Runtime deletion is not used.


In [ ]:
# FIRST CELL: request Google Drive permission before any long operation.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected.')


In [ ]:
# Quiet, Colab-compatible installation. Logs are printed only on failure.
import subprocess, sys

packages = [
    'pandas==2.2.3',
    'transformers==4.57.6',
    'datasets==3.2.0',
    'huggingface-hub==0.36.2',
    'accelerate>=1.10,<2',
    'sentencepiece>=0.2,<0.3',
    'protobuf>=5.29,<6',
    'scikit-learn==1.6.1',
]
proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
if proc.returncode != 0:
    print(proc.stdout)
    raise RuntimeError('Dependency installation failed.')
print('Dependencies installed.')


In [ ]:
import gc, hashlib, json, math, os, random, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub.utils import logging as hf_logging
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as transformers_logging

warnings.filterwarnings('ignore', category=FutureWarning)
hf_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('GPU not found. In Colab select Runtime > Change runtime type > GPU.')

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
except Exception:
    RAM_GB = None

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA autocast dtype:', AMP_DTYPE)
print('RAM (GB):', round(RAM_GB, 1) if RAM_GB else 'unknown')


In [ ]:
# ---------------- LOCKED MATCHED-CONTROL CONFIGURATION ----------------
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
RESULT_DIR = Path('/content/lraa_v11_matched_full_finetuning_results')
PROGRESS_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LCCA Results/lraa_v11_matched_full_finetuning_progress')
PARTIAL_CHECKPOINT_DIR = RESULT_DIR / 'adapter_scorer_checkpoints'
DRIVE_PARTIAL_CHECKPOINT_DIR = PROGRESS_DIR / 'adapter_scorer_checkpoints'

DATA_SEED = 2026
SEEDS = [13, 42, 71]
EPOCHS = 3
MAX_LENGTH = 512
LATENT_RANK = 256
LATENT_HEADS = 8
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
TRAIN_LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
VAL_TARGET = 1075
RESUME_FROM_DRIVE = True

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PARTIAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

required = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [name for name in required if not (DRIVE_MODEL_DIR / name).is_file()]
has_weights = any((DRIVE_MODEL_DIR / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin'))
if missing or not has_weights:
    raise FileNotFoundError(
        f'Incomplete model folder: {DRIVE_MODEL_DIR}; missing={missing}; weights={has_weights}'
    )

LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
MODEL_NAME = str(LOCAL_MODEL_DIR)

locked_config = {
    'experiment': 'lraa_v11_matched_full_finetuning_control',
    'only_changed_factor_relative_to_v9': 'encoder_frozen: true -> false',
    'encoder_frozen': False,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_length': MAX_LENGTH,
    'latent_rank': LATENT_RANK,
    'latent_heads': LATENT_HEADS,
    'batch_size': BATCH_SIZE,
    'evaluation_batch_size': EVAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'learning_rate_all_trainable_parameters': TRAIN_LR,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'passage_grouped_split': True,
    'input_serialization': 'tokenizer(question_prefix_plus_option, passage)',
}
(RESULT_DIR / 'locked_config_before_training.json').write_text(
    json.dumps(locked_config, indent=2), encoding='utf-8'
)
print(json.dumps(locked_config, indent=2))


## Exact V9 data construction

The public BoolQ training split is divided by passage groups; the official
validation split remains untouched for final held-out evaluation. Candidate
order is randomized deterministically during training.


In [ ]:
seed_everything(DATA_SEED)
raw = load_dataset('google/boolq', token=HF_TOKEN)
train_full = raw['train']
heldout_raw = raw['validation']

groups = np.asarray(train_full['passage'])
labels = np.asarray(train_full['answer'], dtype=int)
splitter = GroupShuffleSplit(
    n_splits=80,
    test_size=VAL_TARGET / len(train_full),
    random_state=DATA_SEED,
)
global_rate = labels.mean()
best_split = None
for tr_idx, va_idx in splitter.split(np.zeros(len(labels)), labels, groups):
    score = (
        abs(len(va_idx) - VAL_TARGET) / max(1, VAL_TARGET)
        + abs(labels[va_idx].mean() - global_rate)
    )
    if best_split is None or score < best_split[0]:
        best_split = (score, tr_idx, va_idx)
_, train_idx, val_idx = best_split
train_raw = train_full.select(train_idx.tolist())
val_raw = train_full.select(val_idx.tolist())
assert set(train_raw['passage']).isdisjoint(set(val_raw['passage']))

data_audit = pd.DataFrame([
    {'split': 'train', 'n': len(train_raw), 'yes_rate': np.mean(train_raw['answer']), 'unique_passages': len(set(train_raw['passage']))},
    {'split': 'internal_validation', 'n': len(val_raw), 'yes_rate': np.mean(val_raw['answer']), 'unique_passages': len(set(val_raw['passage']))},
    {'split': 'official_validation_heldout', 'n': len(heldout_raw), 'yes_rate': np.mean(heldout_raw['answer']), 'unique_passages': len(set(heldout_raw['passage']))},
])
data_audit.to_csv(RESULT_DIR / 'data_audit.csv', index=False)
display(data_audit)
assert len(train_raw) == 8351 and len(val_raw) == 1076 and len(heldout_raw) == 3270

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)

def encode_dataset(ds, desc):
    def enc(batch):
        questions = [f'question: {q} answer:' for q in batch['question']]
        yes = tokenizer(
            [q + ' yes' for q in questions], batch['passage'],
            truncation='only_second', max_length=MAX_LENGTH,
        )
        no = tokenizer(
            [q + ' no' for q in questions], batch['passage'],
            truncation='only_second', max_length=MAX_LENGTH,
        )
        return {
            'yes_input_ids': yes['input_ids'],
            'yes_attention_mask': yes['attention_mask'],
            'no_input_ids': no['input_ids'],
            'no_attention_mask': no['attention_mask'],
        }
    return ds.map(
        enc, batched=True, batch_size=256,
        remove_columns=['question', 'passage'], desc=desc,
    )

encoded = DatasetDict({
    'train': encode_dataset(train_raw, 'context train'),
    'validation': encode_dataset(val_raw, 'context validation'),
    'evaluation': encode_dataset(heldout_raw, 'context held-out'),
})
print('Tokenization complete.')


In [ ]:
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, encoded_ds, training=False, seed=0):
        self.ds = encoded_ds
        self.training = training
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[int(idx)]
        swap = False
        if self.training:
            swap = random.Random(
                self.seed * 1000003 + self.epoch * 10007 + int(idx)
            ).random() < 0.5
        semantics = [0, 1] if swap else [1, 0]
        prefix = ['no', 'yes'] if swap else ['yes', 'no']
        candidates = [
            {
                'input_ids': row[f'{p}_input_ids'],
                'attention_mask': row[f'{p}_attention_mask'],
            }
            for p in prefix
        ]
        gold_semantic = int(row['answer'])
        return {
            'candidates': candidates,
            'candidate_semantics': semantics,
            'labels': semantics.index(gold_semantic),
            'semantic_labels': gold_semantic,
        }

def pair_collate(batch):
    flat = [candidate for item in batch for candidate in item['candidates']]
    padded = tokenizer.pad(flat, padding=True, return_tensors='pt')
    bsz = len(batch)
    return {
        'input_ids': padded['input_ids'].view(bsz, 2, -1),
        'attention_mask': padded['attention_mask'].view(bsz, 2, -1),
        'candidate_semantics': torch.tensor([x['candidate_semantics'] for x in batch], dtype=torch.long),
        'labels': torch.tensor([x['labels'] for x in batch], dtype=torch.long),
        'semantic_labels': torch.tensor([x['semantic_labels'] for x in batch], dtype=torch.long),
    }

def make_loader(ds, seed, training=False):
    pair_ds = PairDataset(ds, training=training, seed=seed)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        pair_ds,
        batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE,
        shuffle=training,
        collate_fn=pair_collate,
        num_workers=2,
        pin_memory=True,
        generator=generator,
        persistent_workers=False,
    )
    return pair_ds, loader


## Trainable DeBERTa plus the unchanged LRAA

Gradient checkpointing is an implementation-only memory measure. It
recomputes activations during backward but does not alter the model,
optimizer, batching, or update schedule.


In [ ]:
class LatentRelationAdapter(nn.Module):
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__()
        assert rank % heads == 0
        self.rank = rank
        self.heads = heads
        self.head_dim = rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False)
        self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False)
        self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(
            torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32)
        )

    @property
    def alpha(self):
        return torch.sigmoid(self.alpha_logit)

    def forward(self, hidden, attention_mask, enabled=True):
        if not enabled:
            return hidden
        bsz, seq_len, _ = hidden.shape

        def split(x):
            return x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)

        q = split(self.q(hidden))
        k = split(self.k(hidden))
        v = split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(
            attention_mask[:, None, None, :] == 0,
            torch.finfo(scores.dtype).min,
        )
        attn = torch.softmax(scores, dim=-1)
        z = torch.matmul(attn, v).transpose(1, 2).contiguous()
        z = z.view(bsz, seq_len, self.rank)
        return self.norm(hidden + self.alpha * self.dropout(self.out(z)))

class FullFineTunedLRAA(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        if hasattr(self.encoder.config, 'use_cache'):
            self.encoder.config.use_cache = False
        if hasattr(self.encoder, 'gradient_checkpointing_enable'):
            self.encoder.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={'use_reentrant': False}
            )
        hidden_size = getattr(self.encoder.config, 'hidden_size', None)
        if hidden_size is None:
            hidden_size = getattr(self.encoder.config, 'dim')
        self.hidden_size = int(hidden_size)
        self.adapter = LatentRelationAdapter(
            self.hidden_size, LATENT_RANK, LATENT_HEADS
        )
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(self.hidden_size, 1)

    def score_hidden(self, hidden, attention_mask, adapter_enabled=True):
        hidden = self.adapter(hidden, attention_mask, enabled=adapter_enabled)
        return self.scorer(self.dropout(hidden[:, 0]))

    def forward(self, input_ids, attention_mask, adapter_enabled=True):
        hidden = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        return self.score_hidden(hidden, attention_mask, adapter_enabled)

audit_model = FullFineTunedLRAA()
parameter_counts = {
    'encoder_trainable_parameters': sum(p.numel() for p in audit_model.encoder.parameters() if p.requires_grad),
    'adapter_trainable_parameters': sum(p.numel() for p in audit_model.adapter.parameters() if p.requires_grad),
    'scorer_trainable_parameters': sum(p.numel() for p in audit_model.scorer.parameters() if p.requires_grad),
    'total_trainable_parameters': sum(p.numel() for p in audit_model.parameters() if p.requires_grad),
}
assert parameter_counts['adapter_trainable_parameters'] == 1050625
assert parameter_counts['scorer_trainable_parameters'] == 1025
pd.DataFrame([parameter_counts]).to_csv(RESULT_DIR / 'parameter_counts.csv', index=False)
print(parameter_counts)
del audit_model
gc.collect()


In [ ]:
def semantic_metrics(gold, pred, loss):
    return {
        'loss': float(loss),
        'accuracy': accuracy_score(gold, pred),
        'balanced_accuracy': balanced_accuracy_score(gold, pred),
        'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
        'predicted_yes_rate': float(np.mean(pred)),
        'true_yes_rate': float(np.mean(gold)),
    }

@torch.no_grad()
def evaluate_on_off(model, loader, include_off=True, return_predictions=False):
    model.eval()
    states = ['on', 'off'] if include_off else ['on']
    totals = {state: {'loss_sum': 0.0, 'gold': [], 'pred': []} for state in states}
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE)
        semantics = batch['candidate_semantics']
        gold = batch['semantic_labels'].numpy()
        bsz, nopt, seq_len = ids.shape
        flat_ids = ids.view(bsz * nopt, seq_len)
        flat_mask = mask.view(bsz * nopt, seq_len)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            hidden = model.encoder(
                input_ids=flat_ids, attention_mask=flat_mask
            ).last_hidden_state
            for state in states:
                logits = model.score_hidden(
                    hidden, flat_mask, adapter_enabled=(state == 'on')
                ).view(bsz, nopt)
                loss = F.cross_entropy(logits.float(), labels)
                pred_idx = logits.argmax(dim=-1).cpu()
                semantic_pred = semantics.gather(
                    1, pred_idx[:, None]
                ).squeeze(1).numpy()
                totals[state]['loss_sum'] += loss.item() * bsz
                totals[state]['gold'].extend(gold.tolist())
                totals[state]['pred'].extend(semantic_pred.tolist())

    output = {}
    for state, values in totals.items():
        gold = np.asarray(values['gold'])
        pred = np.asarray(values['pred'])
        result = semantic_metrics(
            gold, pred, values['loss_sum'] / len(gold)
        )
        if return_predictions:
            result['gold'] = gold
            result['pred'] = pred
        output[state] = result
    return output

yes_count = int(np.sum(encoded['train']['answer']))
no_count = len(encoded['train']) - yes_count
class_weight = torch.tensor([
    len(encoded['train']) / (2 * max(no_count, 1)),
    len(encoded['train']) / (2 * max(yes_count, 1)),
], dtype=torch.float32, device=DEVICE)

def cpu_state_dict(module):
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def train_seed(seed):
    seed_everything(seed)
    model = FullFineTunedLRAA().to(DEVICE)
    train_ds, train_loader = make_loader(encoded['train'], seed, training=True)
    _, val_loader = make_loader(encoded['validation'], seed, training=False)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=TRAIN_LR,
        weight_decay=WEIGHT_DECAY,
    )
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_updates = updates_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_updates * WARMUP_RATIO),
        total_updates,
    )
    scaler = torch.amp.GradScaler(
        'cuda', enabled=(AMP_DTYPE == torch.float16)
    )
    optimizer.zero_grad(set_to_none=True)
    best = {'key': None, 'state': None, 'epoch': None, 'metrics': None}
    history = []
    start = time.time()
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch)
        model.train()
        stats = {'loss': 0.0, 'correct': 0, 'seen': 0}
        for step, batch in enumerate(train_loader, 1):
            ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
            labels = batch['labels'].to(DEVICE)
            semantic = batch['semantic_labels'].to(DEVICE)
            bsz, nopt, seq_len = ids.shape
            flat_ids = ids.view(bsz * nopt, seq_len)
            flat_mask = mask.view(bsz * nopt, seq_len)

            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(
                    flat_ids, flat_mask, adapter_enabled=True
                ).view(bsz, nopt)
                per_item = F.cross_entropy(
                    logits.float(), labels, reduction='none'
                )
                loss = (per_item * class_weight[semantic]).mean()

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f'Non-finite loss at seed={seed}, epoch={epoch}, step={step}'
                )
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            stats['loss'] += loss.item() * bsz
            stats['correct'] += (logits.argmax(-1) == labels).sum().item()
            stats['seen'] += bsz

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if step % 250 == 0:
                print(
                    f'Seed {seed}, epoch {epoch}/{EPOCHS}, '
                    f'batch {step}/{len(train_loader)}'
                )

        val = evaluate_on_off(
            model, val_loader, include_off=False
        )['on']
        row = {
            'seed': seed,
            'epoch': epoch,
            'train_loss': stats['loss'] / stats['seen'],
            'train_accuracy': stats['correct'] / stats['seen'],
            **{f'val_{k}': v for k, v in val.items()},
        }
        history.append(row)
        print({k: round(v, 6) if isinstance(v, float) else v for k, v in row.items()})
        key = (val['balanced_accuracy'], val['macro_f1'], -val['loss'])
        if best['key'] is None or key > best['key']:
            best = {
                'key': key,
                'state': cpu_state_dict(model),
                'epoch': epoch,
                'metrics': val,
            }

    model.load_state_dict(best['state'], strict=True)
    model.to(DEVICE).eval()
    del best['state']
    gc.collect()
    return {
        'model': model,
        'best_epoch': best['epoch'],
        'best_val': best['metrics'],
        'history': history,
        'training_seconds': time.time() - start,
        'peak_gpu_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
    }


## Three matched seeds with restart-safe Drive commits

A seed is marked complete only after its held-out predictions, training
history, runtime row, and adapter/scorer state have all been written. An
interrupted seed is retrained rather than silently treated as complete.


In [ ]:
def read_partial(name):
    path = PROGRESS_DIR / name
    if RESUME_FROM_DRIVE and path.is_file():
        return pd.read_csv(path).to_dict('records')
    return []

runtime_rows = read_partial('runtime_partial.csv')
completed_seeds = {int(row['seed']) for row in runtime_rows}

def committed_rows(name):
    return [
        row for row in read_partial(name)
        if int(row['seed']) in completed_seeds
    ]

history_rows = committed_rows('training_history_partial.csv')
run_rows = committed_rows('run_results_partial.csv')
prediction_rows = committed_rows('predictions_partial.csv')

if completed_seeds:
    print('Completed seeds restored from Drive:', sorted(completed_seeds))

for seed in SEEDS:
    partial_name = f'fullft_lraa_adapter_scorer_seed{seed}.pt'
    drive_partial = DRIVE_PARTIAL_CHECKPOINT_DIR / partial_name
    local_partial = PARTIAL_CHECKPOINT_DIR / partial_name
    if seed in completed_seeds:
        if not drive_partial.is_file():
            raise FileNotFoundError(
                f'Seed {seed} is committed but its partial checkpoint is missing: {drive_partial}'
            )
        shutil.copy2(drive_partial, local_partial)
        print(f'Seed {seed} already complete; skipping retraining.')
        continue

    print(f'\n===== MATCHED FULL FINE-TUNING SEED {seed} =====')
    try:
        trained = train_seed(seed)
    except Exception as exc:
        failure = {
            'seed': seed,
            'exception_type': type(exc).__name__,
            'exception_message': str(exc),
            'hyperparameters_changed_after_failure': False,
            'interpretation': (
                'This is an optimization/runtime failure of the locked matched control; '
                'it must not be converted into evidence about adapter contribution.'
            ),
        }
        (RESULT_DIR / 'FAILED_RUN.json').write_text(
            json.dumps(failure, indent=2), encoding='utf-8'
        )
        failure_zip = shutil.make_archive(
            '/content/lraa_v11_matched_full_finetuning_FAILED',
            'zip', root_dir=RESULT_DIR,
        )
        failure_drive = PROGRESS_DIR / Path(failure_zip).name
        shutil.copy2(failure_zip, failure_drive)
        from google.colab import files
        files.download(failure_zip)
        print('Failure artifact backed up to:', failure_drive)
        raise
    model = trained['model']
    _, heldout_loader = make_loader(encoded['evaluation'], seed, training=False)
    heldout = evaluate_on_off(
        model, heldout_loader, include_off=True, return_predictions=True
    )

    for state in ('on', 'off'):
        metric = heldout[state]
        run_rows.append({
            'seed': seed,
            'adapter_state': state,
            'best_epoch': trained['best_epoch'],
            'loss': metric['loss'],
            'accuracy': metric['accuracy'],
            'balanced_accuracy': metric['balanced_accuracy'],
            'macro_f1': metric['macro_f1'],
            'predicted_yes_rate': metric['predicted_yes_rate'],
            'true_yes_rate': metric['true_yes_rate'],
            'adapter_alpha': float(model.adapter.alpha.detach().cpu()),
        })
        for idx, (gold, pred) in enumerate(
            zip(metric['gold'], metric['pred'])
        ):
            prediction_rows.append({
                'seed': seed,
                'adapter_state': state,
                'example_index': idx,
                'gold': int(gold),
                'prediction': int(pred),
            })

    history_rows.extend(trained['history'])
    runtime_rows.append({
        'seed': seed,
        'training_minutes': trained['training_seconds'] / 60,
        'peak_allocated_gpu_memory_gb': trained['peak_gpu_gb'],
    })

    partial_payload = {
        'format_version': 1,
        'experiment': 'lraa_v11_matched_full_finetuning_control',
        'seed': seed,
        'best_epoch': trained['best_epoch'],
        'encoder_state_included': False,
        'warning': 'This small artifact is not independently loadable without the matching full fine-tuned encoder state.',
        'adapter_state_dict': cpu_state_dict(model.adapter),
        'scorer_state_dict': cpu_state_dict(model.scorer),
    }
    torch.save(partial_payload, local_partial)
    shutil.copy2(local_partial, drive_partial)

    # Commit log is written last.
    pd.DataFrame(history_rows).to_csv(PROGRESS_DIR / 'training_history_partial.csv', index=False)
    pd.DataFrame(run_rows).to_csv(PROGRESS_DIR / 'run_results_partial.csv', index=False)
    pd.DataFrame(prediction_rows).to_csv(PROGRESS_DIR / 'predictions_partial.csv', index=False)
    pd.DataFrame(runtime_rows).to_csv(PROGRESS_DIR / 'runtime_partial.csv', index=False)
    completed_seeds.add(seed)
    print(f'Seed {seed} completed and committed to Drive.')

    del trained, model, heldout
    gc.collect()
    torch.cuda.empty_cache()

history_df = pd.DataFrame(history_rows)
run_df = pd.DataFrame(run_rows)
prediction_df = pd.DataFrame(prediction_rows)
runtime_df = pd.DataFrame(runtime_rows)
history_df.to_csv(RESULT_DIR / 'training_history.csv', index=False)
run_df.to_csv(RESULT_DIR / 'adapter_on_off_run_results.csv', index=False)
prediction_df.to_csv(RESULT_DIR / 'adapter_on_off_predictions.csv', index=False)
runtime_df.to_csv(RESULT_DIR / 'runtime_by_seed.csv', index=False)

assert set(run_df.seed.astype(int)) == set(SEEDS)
assert len(run_df) == 6
assert len(prediction_df) == len(SEEDS) * 2 * len(heldout_raw)
display(run_df.sort_values(['seed', 'adapter_state']))


In [ ]:
# Paired estimands and the prespecified training-regime contrast.
T_CRIT_DF2 = 4.302652729911275
FROZEN_BALANCED_GAIN_POINTS = {
    13: 28.498131676170992,
    42: 24.11058282080514,
    71: 30.23795729397838,
}

def mean_ci(values):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    half = T_CRIT_DF2 * sd / math.sqrt(len(values))
    return mean, sd, mean - half, mean + half

def exact_mcnemar_p(a_only, b_only):
    n = a_only + b_only
    if n == 0:
        return 1.0
    k = min(a_only, b_only)
    return min(
        1.0,
        2 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n),
    )

indexed = run_df.set_index(['seed', 'adapter_state']).sort_index()
summary_rows = []
full_gain_vectors = {}
for metric in ('accuracy', 'balanced_accuracy', 'macro_f1'):
    gains = np.array([
        100 * (
            indexed.loc[(seed, 'on'), metric]
            - indexed.loc[(seed, 'off'), metric]
        )
        for seed in SEEDS
    ])
    full_gain_vectors[metric] = gains
    mean, sd, low, high = mean_ci(gains)
    summary_rows.append({
        'metric': metric,
        'adapter_on_mean_percent': 100 * indexed.xs('on', level='adapter_state')[metric].mean(),
        'adapter_off_mean_percent': 100 * indexed.xs('off', level='adapter_state')[metric].mean(),
        'full_finetuning_on_minus_off_points': mean,
        'seed_sd_points': sd,
        'ci95_low_points': low,
        'ci95_high_points': high,
        'positive_seeds': int((gains > 0).sum()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULT_DIR / 'full_finetuning_adapter_ablation_summary.csv', index=False)

full_balanced = full_gain_vectors['balanced_accuracy']
frozen_balanced = np.array([FROZEN_BALANCED_GAIN_POINTS[s] for s in SEEDS])
regime_differences = frozen_balanced - full_balanced
regime_mean, regime_sd, regime_low, regime_high = mean_ci(regime_differences)
regime_df = pd.DataFrame({
    'seed': SEEDS,
    'frozen_v9_on_minus_off_balanced_points': frozen_balanced,
    'matched_fullft_on_minus_off_balanced_points': full_balanced,
    'frozen_minus_fullft_points': regime_differences,
})
regime_df.to_csv(RESULT_DIR / 'frozen_vs_fullft_regime_contrast_by_seed.csv', index=False)

regime_summary = {
    'estimand': 'frozen V9 minus matched full-fine-tuning adapter-on/off balanced-accuracy loss',
    'mean_points': regime_mean,
    'seed_sd_points': regime_sd,
    'ci95_low_points': regime_low,
    'ci95_high_points': regime_high,
    'positive_seeds': int((regime_differences > 0).sum()),
    'support_rule': 'positive in all three seeds and paired seed CI lower bound above zero',
    'support_rule_passed': bool(
        np.all(regime_differences > 0) and regime_low > 0
    ),
}
(RESULT_DIR / 'training_regime_contrast_summary.json').write_text(
    json.dumps(regime_summary, indent=2), encoding='utf-8'
)

mcnemar_rows = []
for seed in SEEDS:
    on = prediction_df[
        (prediction_df.seed.astype(int) == seed)
        & (prediction_df.adapter_state == 'on')
    ]
    off = prediction_df[
        (prediction_df.seed.astype(int) == seed)
        & (prediction_df.adapter_state == 'off')
    ]
    merged = on.merge(
        off, on=['seed', 'example_index', 'gold'], suffixes=('_on', '_off')
    )
    on_correct = merged.prediction_on.to_numpy() == merged.gold.to_numpy()
    off_correct = merged.prediction_off.to_numpy() == merged.gold.to_numpy()
    on_only = int(np.sum(on_correct & ~off_correct))
    off_only = int(np.sum(~on_correct & off_correct))
    mcnemar_rows.append({
        'seed': seed,
        'adapter_on_only_correct': on_only,
        'adapter_off_only_correct': off_only,
        'exact_two_sided_p': exact_mcnemar_p(on_only, off_only),
    })
mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(RESULT_DIR / 'adapter_on_off_mcnemar.csv', index=False)

lines = [
    '| Metric | Adapter on (%) | Adapter off (%) | On−off (points) | 95% seed CI | Positive seeds |',
    '|---|---:|---:|---:|---:|:---:|',
]
for row in summary_rows:
    lines.append(
        f"| {row['metric'].replace('_', ' ')} | "
        f"{row['adapter_on_mean_percent']:.2f} | "
        f"{row['adapter_off_mean_percent']:.2f} | "
        f"{row['full_finetuning_on_minus_off_points']:+.2f} | "
        f"[{row['ci95_low_points']:+.2f}, {row['ci95_high_points']:+.2f}] | "
        f"{row['positive_seeds']}/3 |"
    )
lines.extend([
    '',
    '| Training-regime estimand | Mean difference | 95% seed CI | Positive seeds | Decision |',
    '|---|---:|---:|:---:|:---:|',
    f"| Frozen minus matched full-FT balanced-accuracy ablation loss | {regime_mean:+.2f} | "
    f"[{regime_low:+.2f}, {regime_high:+.2f}] | {int((regime_differences > 0).sum())}/3 | "
    f"{'PASS' if regime_summary['support_rule_passed'] else 'FAIL'} |",
])
markdown_table = '\n'.join(lines)
(RESULT_DIR / 'matched_fullft_control_table.md').write_text(
    markdown_table + '\n', encoding='utf-8'
)
print(markdown_table)
display(regime_df)
display(mcnemar_df)


In [ ]:
# Save exact configuration/environment, verify small artifacts, ZIP, back up, and download.
locked_config.update({
    'train_examples': len(train_raw),
    'internal_validation_examples': len(val_raw),
    'heldout_examples': len(heldout_raw),
    'adapter_parameters': parameter_counts['adapter_trainable_parameters'],
    'scorer_parameters': parameter_counts['scorer_trainable_parameters'],
    'full_encoder_checkpoints_in_download_zip': False,
    'training_regime_support_rule_passed': regime_summary['support_rule_passed'],
})
(RESULT_DIR / 'run_config.json').write_text(
    json.dumps(locked_config, indent=2), encoding='utf-8'
)
environment = {
    'gpu': torch.cuda.get_device_name(0),
    'amp_dtype': str(AMP_DTYPE),
    'torch_version': torch.__version__,
    'transformers_version': __import__('transformers').__version__,
    'datasets_version': __import__('datasets').__version__,
    'pandas_version': pd.__version__,
    'gradient_checkpointing_use_reentrant': False,
}
(RESULT_DIR / 'run_environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)

required_outputs = [
    RESULT_DIR / 'adapter_on_off_run_results.csv',
    RESULT_DIR / 'adapter_on_off_predictions.csv',
    RESULT_DIR / 'training_history.csv',
    RESULT_DIR / 'runtime_by_seed.csv',
    RESULT_DIR / 'full_finetuning_adapter_ablation_summary.csv',
    RESULT_DIR / 'frozen_vs_fullft_regime_contrast_by_seed.csv',
    RESULT_DIR / 'training_regime_contrast_summary.json',
    RESULT_DIR / 'adapter_on_off_mcnemar.csv',
    RESULT_DIR / 'matched_fullft_control_table.md',
    RESULT_DIR / 'run_config.json',
    RESULT_DIR / 'run_environment.json',
] + [
    PARTIAL_CHECKPOINT_DIR / f'fullft_lraa_adapter_scorer_seed{seed}.pt'
    for seed in SEEDS
]
missing = [str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Required result artifacts are missing: {missing}')

manifest_rows = []
for path in sorted(p for p in RESULT_DIR.rglob('*') if p.is_file()):
    if path.name == 'result_file_manifest_sha256.csv':
        continue
    manifest_rows.append({
        'filename': str(path.relative_to(RESULT_DIR)),
        'bytes': path.stat().st_size,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })
pd.DataFrame(manifest_rows).to_csv(
    RESULT_DIR / 'result_file_manifest_sha256.csv', index=False
)

zip_base = '/content/lraa_v11_matched_full_finetuning_control_results'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=RESULT_DIR)
drive_zip_dir = Path('/content/drive/MyDrive/Colab Notebooks/LCCA Results')
drive_zip_dir.mkdir(parents=True, exist_ok=True)
drive_zip_path = drive_zip_dir / Path(zip_path).name
shutil.copy2(zip_path, drive_zip_path)
print('ZIP backup saved to Google Drive:', drive_zip_path)

from google.colab import files
files.download(zip_path)
print('Automatic download triggered:', zip_path)
print('Runtime was NOT disconnected or deleted.')
